<a href="https://colab.research.google.com/github/jmkang21212-wq/projects/blob/master/Colab_%EA%B5%AC%EB%8F%85_%EC%B5%9C%EB%8C%80%ED%95%9C_%ED%99%9C%EC%9A%A9%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets "pillow<12.0" --upgrade

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.1 MB/s eta 0:00:00


In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


In [8]:
!unzip "/content/2026-ssafy-ai-15-2.zip" -d "/content/"

Archive:  /content/2026-ssafy-ai-15-2.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/2026-ssafy-ai-15-2.zip or
        /content/2026-ssafy-ai-15-2.zip.zip, and cannot find /content/2026-ssafy-ai-15-2.zip.ZIP, period.


## 더 빠른 GPU

<p>Colab 유료 요금제 중 하나를 구매한 사용자는 더 빠른 GPU와 더 많은 메모리를 사용할 수 있습니다. 메뉴의 <code>런타임 &gt; 런타임 유형 변경</code>에서 노트북의 GPU 설정을 업그레이드하여 사용 가능 여부에 따라 여러 가속기 옵션 중에서 선택할 수 있습니다.</p>
<p>Colab 무료 버전에서는 NVIDIA T4 GPU를 사용할 수 있으며 할당량 제한 및 가용성이 적용됩니다.</p>

언제든지 다음 셀을 실행하여 할당된 GPU를 확인할 수 있습니다. 아래 코드 셀의 실행 결과가 ‘Not connected to a GPU’인 경우 메뉴의 <code>런타임 &gt; 런타임 유형 변경</code>에서 런타임을 변경하여 GPU 가속기를 사용 설정한 다음 코드 셀을 다시 실행하면 됩니다.

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

노트북에서 GPU를 사용하려면 <code>런타임 &gt; 런타임 유형 변경</code> 메뉴를 선택한 다음 하드웨어 가속기를 원하는 옵션으로 설정하세요.

## 추가 메모리

Colab 유료 요금제 중 하나를 구매한 사용자는 고용량 메모리 VM을 사용할 수 있습니다&#40;사용 가능한 경우&#41;. 더 강력한 GPU는 항상 고용량 메모리 VM과 함께 제공됩니다.
언제든지 다음 코드 셀을 실행하여 사용 가능한 메모리 용량을 확인할 수 있습니다. 아래 코드 셀의 실행 결과가 ‘Not using a high-RAM runtime’인 경우 메뉴의 <code>런타임 &gt; 런타임 유형 변경</code>에서 고용량 RAM 런타임을 사용 설정하고 런타임 구성 전환 버튼에서 고용량 RAM을 선택한 다음 코드 셀을 다시 실행하면 됩니다.

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

## 더 긴 런타임

일정 시간이 지나면 모든 Colab 런타임이 재설정됩니다&#40;런타임에서 코드를 실행하지 않는 경우 더 빠른 방식&#41;. Colab Pro 및 Pro+ 사용자는 Colab 무료 사용자보다 런타임에 더욱 오랫동안 액세스할 수 있습니다.

## 백그라운드 실행

Colab Pro+ 사용자는 백그라운드 실행에 액세스할 수 있으며, 브라우저 탭을 닫은 후에도 노트북이 계속 실행됩니다. 이 기능은 사용 가능한 컴퓨팅 단위가 있는 한 Pro+ 런타임에서 항상 사용 설정됩니다.


## Colab Pro의 리소스 한도 늘리기

Colab에서 제공되는 리소스는 무제한이 아닙니다. Colab을 최대한 활용하려면 필요하지 않은 리소스를 사용하지 않아야 합니다. 예를 들어, 필요할 때만 GPU를 사용하고 완료되면 Colab 탭을 닫는 것이 좋습니다.

한도에 도달하면 사용한 만큼만 지불 요금제를 구매해 컴퓨팅 단위를 추가 구매하여 한도를 늘리세요. 누구나 <a href="https://colab.research.google.com/signup">사용한 만큼만 지불</a>을 통해 컴퓨팅 단위를 구매할 수 있습니다. 구독 없이 사용할 수 있습니다.

## 의견을 보내주세요

<p>의견이 있으면 Google에 보내주세요. 도움말 &gt; '의견 보내기…' 메뉴를 사용하여 간편하게 의견을 제출할 수 있습니다. Colab Pro에서 사용량 한도에 도달하면 Pro+ 구독을 고려해 보세요.</p>
<p>Colab Pro, Pro+, 사용한 만큼만 지불 요금제에서 결제 관련 오류나 기타 문제가 발생하면 <a href="mailto:colab-billing@google.com">colab-billing@google.com</a>으로 이메일을 보내 문의해 주시기 바랍니다.</p>

## 추가 리소스

### Colab에서 메모장 사용하기
- [Colab 개요](/notebooks/basic_features_overview.ipynb)
- [Markdown 가이드](/notebooks/markdown_guide.ipynb)
- [라이브러리 가져오기 및 종속 항목 설치하기](/notebooks/snippets/importing_libraries.ipynb)
- [GitHub에서 노트 저장 및 로드하기](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb)
- [대화형 양식](/notebooks/forms.ipynb)
- [대화형 위젯](/notebooks/widgets.ipynb)

<a name="working-with-data"></a>
### 데이터로 작업하기
- [데이터 로드: 드라이브, 스프레드시트, Google Cloud Storage](/notebooks/io.ipynb)
- [차트: 데이터 시각화하기](/notebooks/charts.ipynb)
- [BigQuery 시작하기](/notebooks/bigquery.ipynb)

### 머신러닝 단기집중과정
다음은 Google 온라인 머신러닝 과정에서 가져온 일부 메모장입니다. <a href="https://developers.google.com/machine-learning/crash-course/">전체 과정 웹사이트</a>에서 자세한 내용을 확인하세요.
- [Pandas DataFrame 소개](https://colab.research.google.com/github/google/eng-edu/blob/main/ml/cc/exercises/pandas_dataframe_ultraquick_tutorial.ipynb)
- [Pandas를 가속화하는 RAPIDS cuDF 소개](https://nvda.ws/rapids-cudf)
- [cuML의 가속기 모드 시작하기](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/getting_started_tutorials/cuml_sklearn_colab_demo.ipynb)


<a name="using-accelerated-hardware"></a>
### 가속 하드웨어 사용하기
- [Flax NNX API를 사용하여 MNIST 데이터 세트에서 필기 입력 숫자를 분류하도록 CNN 학습시키기](https://colab.research.google.com/github/google/flax/blob/main/docs_nnx/mnist_tutorial.ipynb)
- [JAX로 이미지 분류를 위한 Vision Transformer&#40;ViT&#41; 학습시키기](https://colab.research.google.com/github/jax-ml/jax-ai-stack/blob/main/docs/source/JAX_Vision_transformer.ipynb)
- [JAX를 사용한 Transformer 언어 모델 기반 텍스트 분류](https://colab.research.google.com/github/jax-ml/jax-ai-stack/blob/main/docs/source/JAX_transformer_text_classification.ipynb)

<a name="machine-learning-examples"></a>

## 머신러닝 예시

일부 추천 예시는 다음과 같습니다.

- <a href="https://docs.jaxstack.ai/en/latest/JAX_for_LLM_pretraining.html">JAX AI 스택으로 miniGPT 언어 모델 학습시키기</a>
- <a href="https://github.com/google/tunix/blob/main/examples/qlora_gemma.ipynb">Tunix를 사용한 LLM용 LoRA/QLoRA 미세 조정</a>
- <a href="https://keras.io/examples/keras_recipes/parameter_efficient_finetuning_of_gemma_with_lora_and_qlora/">LoRA 및 QLoRA를 사용한 Gemma의 Parameter-Efficient Fine-Tuning&#40;PEFT&#41;</a>
- <a href="https://keras.io/keras_hub/guides/hugging_face_keras_integration/">Hugging Face Transformers 체크포인트 로드</a>
- <a href="https://keras.io/guides/int8_quantization_in_keras/">Keras의 8비트 정수 양자화</a>
- <a href="https://keras.io/examples/keras_recipes/float8_training_and_inference_with_transformer/">간단한 Transformer 모델을 사용한 Float8 학습 및 추론</a>
- <a href="https://keras.io/keras_hub/guides/transformer_pretraining/">KerasHub로 처음부터 Transformer 사전 학습시키기</a>
- <a href="https://keras.io/examples/vision/mnist_convnet/">단순 MNIST 합성곱 신경망</a>
- <a href="https://keras.io/examples/vision/image_classification_from_scratch/">Keras 3를 사용해 처음부터 이미지 분류 구현</a>
- <a href="https://keras.io/keras_hub/guides/classification_with_keras_hub/">KerasHub를 사용한 이미지 분류</a>
